<!-- source: new + slide 42–44 -->
# M4 · Dane tabelaryczne: SQL, Genie Agent i kontrola dostępu

**Przebieg:** prezentacja, demo, lab

**„Ile mamy klientów VIP?” Dwie poprawne odpowiedzi:**

| Z dokumentów (RAG, M3) | Z tabeli (funkcja albo Genie) |
|---|---|
| „Według raportu o segmentach klienci VIP to około 9,5 tys. firm z najniższym recency… [01_segmentacja_klientow #1]” | **9 541** |
| narracja, stan z dnia wygenerowania raportu, kontekst i wnioski | liczba, stan na teraz, zero interpretacji |

Pytanie „ile” idzie do tabeli, a pytanie „dlaczego” do dokumentów. Agent musi to rozróżniać i uczymy go tego opisami narzędzi.

**Trzy drogi do tej samej tabeli:**
- **SQL wprost:** pełna swoboda, zero kontroli. Dla człowieka, nie dla agenta.
- **Funkcja Unity Catalog** (M2): stałe pytanie, stały kształt odpowiedzi, bez PII. Kontrakt między danymi a agentem.
- **Genie Agent** (dawniej Genie Space): pytanie po polsku → wygenerowany SQL → wynik. Do pytań ad hoc, których nie przewidziały funkcje.

| Część | Co robisz | Lab |
|---|---|---|
| 1 | oczekiwane wyniki czterech pytań z SQL | 2 z 4 zapytań |
| 2 | Genie Agent w UI i porównanie z RAG | UI |
| 3 | uprawnienia, row filter (demo), column mask | wyrażenie maski |
| 4 | **obowiązkowe** zdjęcie filtra i maski przed M5 | — |

In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
# source: WS3[2]
dbutils.library.restartPython()

In [ ]:
# source: new + WS4[3] + WS2[6]
# Wspólna konfiguracja warsztatu — ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

In [ ]:
# source: new
import re
import time

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
USERNAME = spark.sql("SELECT current_user()").first()[0]
print(f"Użytkownik: {USERNAME} | tabela: {GOLD_TABLE} ({spark.table(GOLD_TABLE).count():,} wierszy)")

<!-- source: WS2[36] + WS3[20] -->
## 1. Odpowiedź z tabeli: oczekiwane wyniki

Zanim zapytasz Genie, ustal, **co jest poprawną odpowiedzią**. Te same liczby są w decku, w Przewodniku i w scorerach ewaluacji. To pierwsze wiersze zestawu testowego, który w produkcji uruchamia się przed każdą zmianą.

**Lab:** dopisz dwa brakujące zapytania. Komórka pomija te, które mają jeszcze `...`.

In [ ]:
# source: WS3[20] + WS2[37]
EXPECTED_SQL = {
    "Ile mamy klientów VIP (loyalty_segment = 3)?":
        f"SELECT COUNT(*) AS vip FROM {GOLD_TABLE} WHERE loyalty_segment = 3",
    "Jaki stan ma najwięcej klientów?":
        f"SELECT state, COUNT(*) AS klienci FROM {GOLD_TABLE} GROUP BY state ORDER BY klienci DESC LIMIT 1",
    "Ile klientów nie złożyło żadnego zamówienia?":
        f"SELECT COUNT(*) AS bez_zamowien FROM {GOLD_TABLE} WHERE num_orders = 0",
    "Jaka jest średnia wartość monetary dla segmentu VIP?":
        f"SELECT ROUND(AVG(monetary), 2) AS avg_monetary FROM {GOLD_TABLE} WHERE loyalty_segment = 3",
}

expected_values = {}
for question, query in EXPECTED_SQL.items():
    if query is ...:
        print(f"⏭  {question}  (TODO)")
        continue
    expected_values[question] = spark.sql(query).first().asDict()
    print(f"✅ {question}  →  {expected_values[question]}")
# Oczekiwane: 9 541 VIP · NY 3 417 · 26 862 bez zamówień · 1038.72

<!-- source: slide 46 -->
### Kiedy funkcja Unity Catalog, a kiedy Genie

| Kryterium | Genie Agent | Funkcja Unity Catalog |
|---|---|---|
| Styl zapytania | język naturalny → SQL nad wskazanymi tabelami | parametry do gotowej, przewidywalnej logiki |
| Elastyczność | wysoka: pytania otwarte | celowo ograniczona |
| Najlepsze do | analityka eksploracyjna, pytania ad hoc | powtarzalne pobranie danych, logika biznesowa |
| Dostęp dla agenta | zarządzany serwer MCP `/api/2.0/mcp/genie/{id}` (M6) | `EXECUTE` na funkcji, test payloadem bez modelu |
| U nas dziś | UI i Tool w Playground | trzy funkcje w agencie (M2, M5) |

Z kursu Databricks: *komplementarne, nie konkurencyjne*. Supervisor pyta Genie „co napędzało wzrost w Q4?”, a funkcję „pokaż klienta 1234”. „Co mówią dokumenty” idzie do RAG.

<!-- source: WS1[37] + slide 45 + slide 48 -->
## 2. Lab: Genie Agent nad tabelą Gold

1. W lewym pasku **Genie → New** (w nowym UI *Genie Agent*, dawniej *Genie Space*).
2. **Tytuł:** `Retail Customer Intelligence Assistant` (dokładnie tak, bo M6 i ewaluacja szukają tej nazwy). **Dane:** `workspace.default.gold_customer_360`. **Warehouse:** jedyny Serverless na Free Edition.
3. **Instructions:**
   ```text
   Odpowiadaj po polsku. Główna tabela: workspace.default.gold_customer_360.
   loyalty_segment: 0=Nowi/Nieaktywni, 1=Rozwijający się, 2=Regularni, 3=VIP.
   tax_id i customer_name to PII — nie pokazuj ich wartości.
   ```
4. **Sample questions:** „Ile mamy klientów VIP (loyalty_segment = 3)?”, „Który stan ma najwięcej klientów i jaką średnią wartość monetary?”, „Jaki procent klientów nie złożył żadnego zamówienia (num_orders = 0)?”, „Porównaj średni recency_days i monetary między segmentami”.
5. Zadaj Genie **cztery pytania z części 1** i porównaj z oczekiwanymi wynikami. Rozwiń **Show code**, żeby zobaczyć wygenerowany SQL.
6. Porównaj z M3: to samo pytanie w RAG dało narrację z cytatem, a w Genie liczbę i SQL. Które kiedy wybrać?

| Pytanie | Oczekiwane (SQL) | Genie | RAG (M3) |
|---|---|---|---|
| VIP | 9 541 | | |
| stan z największą liczbą klientów | NY, 3 417 | | |
| bez zamówień | 26 862 | | |
| średnia monetary VIP | 1038,72 | | |

In [ ]:
# source: WS2[39–41]
# Genie Agent z kodu: to samo pytanie przez SDK. Na Free Edition API ma limit ok. 5 pytań na minutę.
def find_genie_space_id(title: str) -> str | None:
    for space in w.genie.list_spaces().spaces or []:
        if title == (space.title or ""):
            return space.space_id
    return None


GENIE_SPACE_ID = find_genie_space_id(GENIE_TITLE)
if not GENIE_SPACE_ID:
    print(f"Nie znaleziono Genie Agenta „{GENIE_TITLE}”. Utwórz go w UI (krok 2 wyżej).")
else:
    for question in list(EXPECTED_SQL)[:2]:
        message = w.genie.start_conversation_and_wait(space_id=GENIE_SPACE_ID, content=question)
        print(f"❓ {question}")
        for attachment in message.attachments or []:
            if attachment.query:
                print(f"   SQL: {attachment.query.query}")
                result = w.genie.get_message_attachment_query_result(
                    GENIE_SPACE_ID, message.conversation_id, message.message_id, attachment.attachment_id
                )
                print(f"   wynik: {result.statement_response.result.data_array[:3]}")
            if attachment.text:
                print(f"   💬 {attachment.text.content}")
        time.sleep(13)

<!-- source: slide 47 + WS2[17] + WS2[20] + slide 64 -->
## 3. Dostęp do danych w kontrolowany sposób: dwie warstwy

| W narzędziu (M2) | W danych (Unity Catalog, teraz) |
|---|---|
| funkcja zwraca tylko wybrane kolumny, bez `tax_id` | **column mask**: `tax_id` zamaskowany wszędzie — w SQL, w Genie i w funkcjach |
| agent nie dostaje tabeli, tylko funkcję | **row filter**: użytkownik widzi tylko swoje stany |
| `GRANT EXECUTE` na funkcji decyduje, kto ją wywoła | działa niezależnie od modelu, promptu i narzędzia |

**Obrona w głąb:** prompt może zawieść i funkcja może zawieść, ale maska w katalogu nie. Compliance Officer śpi spokojnie dopiero po trzeciej warstwie.

**Uprawnienia (least privilege).** Minimum dla agenta z M5, gdy działa jako service principal:

```sql
GRANT EXECUTE ON FUNCTION workspace.default.get_average_customer_value TO `retail-agent-sp`;
GRANT EXECUTE ON FUNCTION workspace.default.get_customer_profile       TO `retail-agent-sp`;
GRANT EXECUTE ON FUNCTION workspace.default.format_customer_for_agent  TO `retail-agent-sp`;
GRANT SELECT  ON TABLE    workspace.default.retail_rag_chunks_index    TO `retail-agent-sp`;
-- Żadnego SELECT na gold_customer_360: agent dostaje funkcje, nie tabelę.
```

Na Free Edition jesteś jedynym użytkownikiem i właścicielem obiektów, więc `GRANT` niczego nie odbierze Tobie. Dlatego filtr i maskę poniżej piszemy z warunkiem opartym na **grupie, do której nie należysz**. Zobaczysz dokładnie to, co zobaczyłby analityk bez uprawnień.

In [ ]:
%sql
-- source: WS2[21]
SHOW GRANTS ON TABLE workspace.default.gold_customer_360

<!-- source: WS2[24] -->
### Row filter: które wiersze widzi użytkownik

Row filter to funkcja SQL zwracająca `BOOLEAN`. Unity Catalog wstrzykuje ją do **każdego** zapytania na tabeli: z notebooka, z Genie, z dashboardu i z funkcji agenta. Wiersz jest widoczny tylko wtedy, gdy funkcja zwraca `TRUE`.

**Scenariusz:** analityk regionu Kalifornia widzi tylko klientów z CA, a pełny obraz ma grupa `all_states_analysts`. W pierwszej wersji WS2 warunek sprawdzał grupę, do której należy każdy użytkownik, więc był zawsze prawdziwy i filtr niczego nie ukrywał. Tu warunek zależy od grupy, której w Twoim workspace nie ma.

**Demo prowadzącego (Mariusz):** uruchom komórkę razem z prowadzącym i sprawdź wynik. Predykat jest gotowy.

In [ ]:
%sql
-- source: WS2[25] + WS2[26]
CREATE OR REPLACE FUNCTION workspace.default.retail_row_filter(state_val STRING)
RETURNS BOOLEAN
COMMENT 'Row filter warsztatu: wszystkie stany widzi grupa all_states_analysts, pozostali tylko CA.'
RETURN is_account_group_member('all_states_analysts') OR state_val = 'CA';

ALTER TABLE workspace.default.gold_customer_360
SET ROW FILTER workspace.default.retail_row_filter ON (state);

-- Test: zostaje tylko CA
SELECT state, COUNT(*) AS klienci
FROM workspace.default.gold_customer_360
GROUP BY state
ORDER BY klienci DESC;

<!-- source: WS2[27] -->
### Column mask: jakie wartości kolumny widzi użytkownik

Row filter ukrywa **całe wiersze**, a column mask ukrywa **wartość w kolumnie**. Wiersz jest widoczny, ale `tax_id` pokazuje `***MASKED***`. Maska zwraca ten sam typ co kolumna i działa także w Genie.

**Lab:** napisz wyrażenie `CASE`: prawdziwą wartość widzi tylko grupa `compliance_officers`.

In [ ]:
%sql
-- source: WS2[28] + WS2[30]
CREATE OR REPLACE FUNCTION workspace.default.mask_tax_id(tax_id_val STRING)
RETURNS STRING
COMMENT 'Column mask warsztatu: prawdziwy tax_id widzi tylko grupa compliance_officers.'
RETURN CASE WHEN is_account_group_member('compliance_officers') THEN tax_id_val ELSE '***MASKED***' END;

ALTER TABLE workspace.default.gold_customer_360
ALTER COLUMN tax_id SET MASK workspace.default.mask_tax_id;

-- Test: filtr i maska działają razem (tylko CA, tax_id zamaskowany)
SELECT customer_id, state, loyalty_segment, tax_id
FROM workspace.default.gold_customer_360
WHERE tax_id IS NOT NULL
LIMIT 5;

<!-- source: slide 47 -->
**Sprawdź w Genie:** zapytaj swojego Genie Agenta „Pokaż tax_id pięciu klientów” i „Ilu mamy klientów w NY?”. Genie wykonuje SQL z **Twoją** tożsamością, więc filtr i maska obowiązują także tam: zobaczysz `***MASKED***` i 0 klientów w NY.

To jest sedno warstwy danych: nie trzeba nic zmieniać w Genie, w funkcjach ani w promptach.

<!-- source: WS2[24] + WS2[27] -->
## Demo prowadzącego: dwie tożsamości na Premium

Na workspace Premium prowadzący pokazuje ten sam `SELECT` z dwóch kont:
1. **Konto A** (członek grup `all_states_analysts` i `compliance_officers`): wszystkie stany, prawdziwe `tax_id`.
2. **Konto B** (bez tych grup): tylko CA, `***MASKED***`.
3. To samo pytanie w Genie Agent z obu kont daje dwie różne odpowiedzi z jednej tabeli i jednej polityki.

Grupy zakłada się w **Settings → Identity and access → Groups**. Filtr i maska nie wymagają zmian.

<!-- source: WS2[31] -->
## 4. Obowiązkowo przed M5: zdejmij filtr i maskę

Agent z M5 ma odpowiadać o wszystkich stanach. Z aktywnym filtrem widziałby tylko CA, a macierz tras dałaby fałszywe wyniki. Komórka poniżej zdejmuje filtr i maskę, usuwa funkcje i sprawdza, że tabela wróciła do 28 813 wierszy.

W produkcji nie zdejmuje się zabezpieczeń bez przeglądu: `SHOW GRANTS`, `DESCRIBE TABLE EXTENDED` (sekcje *Row Filter* i *Column Masks*) oraz `information_schema`.

In [ ]:
# source: WS2[32] + new
for statement in [
    f"ALTER TABLE {GOLD_TABLE} DROP ROW FILTER",
    f"ALTER TABLE {GOLD_TABLE} ALTER COLUMN tax_id DROP MASK",
    f"DROP FUNCTION IF EXISTS {CATALOG}.{SCHEMA}.retail_row_filter",
    f"DROP FUNCTION IF EXISTS {CATALOG}.{SCHEMA}.mask_tax_id",
]:
    try:
        spark.sql(statement)
        print(f"✅ {statement}")
    except Exception as e:
        print(f"ℹ️  {statement} → {str(e)[:120]}")

rows = spark.table(GOLD_TABLE).count()
sample_tax_id = spark.table(GOLD_TABLE).where("tax_id IS NOT NULL").select("tax_id").first()[0]
assert rows == 28_813, f"Tabela ma {rows} wierszy — row filter nadal aktywny?"
assert re.fullmatch(r"\d{2}-\d{7}", sample_tax_id), "tax_id nadal zamaskowany — sprawdź DROP MASK"
print(f"\n✅ {GOLD_TABLE}: {rows:,} wierszy, tax_id bez maski. Możesz przejść do M5.")

In [ ]:
# source: WS3[36] + WS2[42]
import json
import os
from pathlib import Path

baseline_path = Path(os.getcwd()).parent / "data" / "evaluation" / "genie_baseline_scores.json"
if not baseline_path.exists():
    print("Brak genie_baseline_scores.json — prowadzący nie uruchomił kroku 8 w prepare_data_premium.")
else:
    baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
    print(f"Genie Agent „{baseline['space_title']}”, ewaluacja z {baseline['evaluated_at']} ({baseline['n_cases']} pytań)\n")
    for scorer_name in ("safety", "correctness", "no_pii_leak", "retail_domain"):
        value = baseline["scores"].get(scorer_name)
        print(f"  {scorer_name:<15} {'—' if value is None else f'{value:.0%}'}")
    print("\nGenie wygrywa na liczbach (correctness), RAG i Knowledge Assistant na narracji z cytatami.")

<!-- source: slide 48 -->
### (jeśli zostanie czas) Genie Agent jako Tool w Playground

**Playground → Tools → Add tool → Genie** → `Retail Customer Intelligence Assistant`. Dodaj obok funkcje z M2. Zapytaj o coś, czego funkcje nie przewidziały, np. *„Który stan ma najwyższą średnią wartość klienta w segmencie Regularni?”*. Które narzędzie wybrał model? Rozwiń panel i zobacz SQL wygenerowany przez Genie.

<!-- source: new -->
## Poziomy 2 i 3: kiedy skończysz ścieżkę

| Poziom | Zadanie |
|---|---|
| **2. Transfer** | Komórka poniżej: maska na `cardNumber` w kopii danych Bakehouse. Potem utwórz Genie Agenta na `bh_transactions` i zapytaj o numery kart, a potem o sprzedaż per franczyza. |
| **3. Wyzwanie** | Genie Agent na tabeli Airbnb (`data/practice`) z instrukcjami i 3 przykładowymi zapytaniami SQL (**Instructions → SQL queries**). Wyzwanie: maska zależna od grupy + row filter na regionie w tej samej tabeli i sprawdzenie, co widzi Genie. |

In [ ]:
# source: new + WS2[28]
# Poziom 2: maska na numerze karty w kopii danych Bakehouse (samples jest tylko do odczytu)
BH_TABLE = f"{CATALOG}.{SCHEMA}.bh_transactions"
spark.sql(f"CREATE OR REPLACE TABLE {BH_TABLE} AS SELECT * FROM samples.bakehouse.sales_transactions")
card_type = spark.table(BH_TABLE).schema["cardNumber"].dataType.simpleString()
masked_value = "'****'" if card_type == "string" else f"CAST(NULL AS {card_type})"

spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.bh_mask_card(card {card_type})
    RETURNS {card_type}
    COMMENT 'Column mask: payment card numbers visible only to the payments_team group.'
    RETURN CASE WHEN is_account_group_member('payments_team') THEN card ELSE {masked_value} END
""")
spark.sql(f"ALTER TABLE {BH_TABLE} ALTER COLUMN cardNumber SET MASK {CATALOG}.{SCHEMA}.bh_mask_card")
display(spark.table(BH_TABLE).select("transactionID", "franchiseID", "paymentMethod", "cardNumber").limit(5))

<!-- source: new + slide 47 -->
## Karta wzorca: kontrolowany dostęp do danych

1. **Wypisz dane wrażliwe** swojej domeny i zdecyduj: nie kopiować, maskować czy filtrować wiersze.
2. **Funkcja dla agenta** zwraca tylko potrzebne kolumny; **maska i filtr** w Unity Catalog działają wszędzie (SQL, Genie, funkcje).
3. **Warunek na grupie**, nie na użytkowniku; sprawdź z konta bez uprawnień.
4. **Genie** do pytań ad hoc, **funkcja** do powtarzalnych: komplementarne, nie konkurencyjne.

**Canvas agenta** (`workshop/transfer/canvas_agenta.md`): dla każdej kolumny wrażliwej zapisz decyzję (nie kopiuj / maska / filtr) i grupę, która widzi wartość.

<!-- source: new -->
## Podsumowanie

- Te same pytania mają **dwie poprawne odpowiedzi**: liczbę na teraz z tabeli i narrację ze snapshotu raportu. Agent musi wiedzieć, której szuka.
- **Genie Agent** odpowiada na pytania ad hoc przez wygenerowany SQL. **Funkcja UC** to przewidywalny kontrakt dla agenta. Są komplementarne.
- Kontrola dostępu ma dwie warstwy: **narzędzie bez PII** oraz **row filter i column mask** w Unity Catalog, które działają wszędzie, także w Genie.
- Least privilege dla agenta: `EXECUTE` na funkcjach, `SELECT` na indeksie, **bez** `SELECT` na tabeli.

**Dalej:** M5. Składamy agenta z czterech narzędzi i sprawdzamy, czy wybiera właściwą trasę.